In [1]:
import matplotlib.pyplot as plt
import torch
from nnfabrik.builder import get_data, get_trainer

from model import stacked_core_full_gauss_readout
from trainer import standard_trainer

from autoencoder import Autoenc

random_seed = 42

device = "cuda:7"
torch.cuda.set_device(device)

In [2]:
# loading the SENSORIUM+ dataset
filenames = [
    "/srv/user/polina/sensorium/sensorium/notebooks/data/static23343-5-17-GrayImageNet-94c6ff995dac583098847cfecd43e7b6.zip",
    #"/srv/user/polina/sensorium/sensorium/notebooks/data/static21067-10-18-GrayImageNet-94c6ff995dac583098847cfecd43e7b6.zip", 

]
data_key = '23343-5-17'

dataset_fn = "sensorium.datasets.static_loaders"
dataset_config = {
    "paths": filenames,
    "normalize": True,
    "exclude": None,
    "include_behavior": False,
    "include_eye_position": False,
    "batch_size": 128,
    "scale": 0.25,
}

dataloaders = get_data(dataset_fn, dataset_config)

In [3]:
model_config = {
    "pad_input": False,
    "stack": -1,
    "layers": 4,
    "input_kern": 9,
    "gamma_input": 6.3831,
    "gamma_readout": 0.0076,
    "hidden_kern": 7,
    "hidden_channels": 64,
    "depth_separable": True,
    "grid_mean_predictor": {
        "type": "cortex",
        "input_dimensions": 2,
        "hidden_layers": 1,
        "hidden_features": 30,
        "final_tanh": True,
    },
    "init_sigma": 0.1,
    "init_mu_range": 0.3,
    "gauss_type": "full",
    "shifter": False, 
    "autoencoder": None, 
}

model = stacked_core_full_gauss_readout(dataloaders, random_seed, **model_config)
model.load_state_dict(torch.load('model_weights.pth'))

/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:74: UserWarning: Use of 'gamma_readout' is deprecated. Use 'feature_reg_weight' instead. If 'feature_reg_weight' is defined, 'gamma_readout' is ignored
  warnings.warn(
/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:95: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


<All keys matched successfully>

In [4]:
checkpoint = torch.load('model_weights.pth')
features = checkpoint['readout.23343-5-17._features']
features = features.squeeze().permute(1, 0).detach()

In [5]:
hidden_dims = 1024
hidden_layers = 5
hidden_dims = [hidden_dims] * hidden_layers
autoencoder = Autoenc(64, 2, hidden_dims, batch_norm=True)
autoencoder.to(device)

Autoenc(
  (encoder): Sequential(
    (0): Linear(in_features=64, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=1024, out_features=1024, bias=True)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=1024, bias=True)
    (7): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Linear(in_features=1024, out_features=1024, bias=True)
    (10): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
    (12): Linear(in_features=1024, out_features=1024, bias=True)
    (13): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Linear(in_features=1024, out_features=2, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(i

In [6]:
criterion = torch.nn.MSELoss(reduction='sum')
optim = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=0.3, patience=100)

epochs = int(1e15)
min_lr = 1e-5

for epoch in range(epochs):
    recon = autoencoder(features)
    loss = criterion(features, recon)

    optim.zero_grad()
    loss.backward()
    optim.step()
    scheduler.step(loss.item())
    if epoch % 100 == 0:
        lr = optim.param_groups[0]['lr']
        if lr < min_lr:
            break

        print(lr, loss.item())

0.001 95201.78125
0.001 13372.58203125
0.001 12467.2666015625
0.001 11665.017578125
0.001 11151.09765625
0.001 10666.953125
0.001 10579.75390625
0.001 10022.9580078125
0.001 9656.962890625
0.001 9386.1611328125
0.001 9406.4208984375
0.001 9017.361328125
0.001 8893.7490234375
0.001 8533.12890625
0.001 8334.990234375
0.001 8300.5478515625
0.001 7893.587890625
0.001 7789.1376953125
0.001 7548.1640625
0.001 7439.86376953125
0.001 7456.783203125
0.001 7144.408203125
0.001 7067.134765625
0.001 6934.0888671875
0.001 6533.50634765625
0.001 6548.779296875
0.001 6334.427734375
0.001 6295.34326171875
0.001 6372.87451171875
0.001 5918.99365234375
0.001 5813.224609375
0.001 5814.88037109375
0.001 5694.10400390625
0.001 5771.4228515625
0.001 5622.6923828125
0.001 5484.330078125
0.001 5644.5732421875
0.001 5272.35107421875
0.0003 4389.2607421875
0.0003 3966.538818359375
0.0003 3857.288818359375
0.0003 3760.18994140625
0.0003 3683.958984375
0.0003 3657.777587890625
0.0003 3646.34423828125
0.0003 3593.

In [7]:
model_autoenc = stacked_core_full_gauss_readout(dataloaders, random_seed, **model_config)
model_autoenc.load_state_dict(torch.load('model_weights.pth'))
model_autoenc.readout[data_key].autoencoder = autoencoder

In [8]:
from sensorium.utility.scores import get_correlations

model.eval()

# Compute avg validation and test correlation
print(get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
))

model_autoenc.eval()

# Compute avg validation and test correlation
print(get_correlations(
    model_autoenc, dataloaders["validation"], device=device, as_dict=False, per_neuron=False 
))

0.32743725
0.31117892


In [9]:
batch = next(iter(dataloaders["validation"][data_key]))
imgs, responses = batch

In [10]:
model(imgs.cpu())

tensor([[0.3629, 0.5861, 0.3062,  ..., 0.7069, 0.3322, 0.6246],
        [0.1037, 0.0694, 0.0654,  ..., 0.0774, 0.1657, 0.0403],
        [0.2231, 0.2489, 0.1406,  ..., 0.4511, 0.4467, 0.7163],
        ...,
        [0.4369, 1.2152, 1.0273,  ..., 0.4265, 0.4621, 0.2309],
        [0.8007, 1.3410, 0.5722,  ..., 0.2958, 0.5693, 0.4758],
        [0.3330, 0.5028, 0.1791,  ..., 0.1497, 0.5552, 0.5729]],
       grad_fn=<AddBackward0>)

In [11]:
model_autoenc(imgs.cpu())

tensor([[0.2782, 0.4985, 0.2735,  ..., 0.3390, 0.3363, 0.3087],
        [0.0918, 0.0991, 0.0915,  ..., 0.0899, 0.1102, 0.0440],
        [0.2521, 0.2165, 0.1472,  ..., 0.3649, 0.4806, 0.7780],
        ...,
        [0.4498, 1.4377, 1.1559,  ..., 0.3309, 0.6459, 0.2115],
        [0.7253, 1.1005, 0.5456,  ..., 0.2178, 0.5712, 0.3202],
        [0.3122, 0.5873, 0.1513,  ..., 0.1560, 0.3179, 0.4522]],
       grad_fn=<AddBackward0>)